In [ ]:
# 原方法
def calculate_initial_displacement(N, file_path, dataset, nodes):
    """
    Calculate the displacement for a given system of order method.

    Parameters:
    N (int): The number of nodes in the system.
    file_path (str): Path to the stiffness matrix file.
    dataset (DataFrame): A pandas DataFrame containing the necessary data.
    nodes (array): Array of node positions.

    Returns:
    ndarray: The displacement array.
    """

    # Extract data from the dataset
    added_mass = dataset['added_mass'][0].values
    radiation_damping = dataset['radiation_damping'][0].values
    inertia_matrix = dataset['inertia_matrix'].values
    hydrostatic_stiffness = dataset['hydrostatic_stiffness'].values
    F_w = dataset['Froude_Krylov_force'][0].values + dataset['diffraction_force'][0].values

    # Construct the combined matrices
    M = added_mass + inertia_matrix  # Total mass
    C = radiation_damping  # Damping
    K = hydrostatic_stiffness  # Stiffness

    # Insert matrices into the system
    mass = DM_A.insert_matrix(N, M, nodes)
    damping = DM_A.insert_matrix(N, C, nodes)
    hy_stiffness = DM_A.insert_matrix(N, K, nodes)

    # Assemble the stiffness matrix
    stiffness = dm_r.get_stiffness_matrix(file_path) + hy_stiffness

    # Assemble the force matrix
    K_F_w = DM_A.extend_force_matrix(F_w, nodes, N)
    omega = dataset.omega.values
    # Solve in the frequency domain
    X = DM_A.solve_frequency_domain(mass, damping, stiffness, K_F_w, omega)

    return X


In [ ]:
def perform_expansion_and_solve(num_nodes, node_position_params, hydrodynamic_data_path, structure_data_paths):
    """
    Perform expansion process and solve for global displacement in the frequency domain.

    Args:
        num_nodes (int): Total number of nodes in the model.
        node_position_params (tuple): Parameters used to calculate the positions of master nodes.
        hydrodynamic_data_path (str): File path to the hydrodynamic data stored in NetCDF format.
        structure_data_paths (dict): Dictionary with file paths to the mass and stiffness matrices.

    Returns:
        ndarray: The reordered global displacement array after performing dynamic analysis.
    """

    # Calculate node positions for the master nodes
    master_nodes = DM_A.calculate_node_positions(*node_position_params)

    # Load and process the hydrodynamic dataset
    dataset = merge_complex_values(xr.open_dataset(hydrodynamic_data_path))
    omega = dataset.omega.values  # Extract angular frequency from dataset

    # Load mass and stiffness matrices
    M = dm_r.get_stiffness_matrix(structure_data_paths['mass'])
    k = dm_r.get_stiffness_matrix(structure_data_paths['stiffness'])
    
    # Reduce the degrees of freedom in the mass and stiffness matrices
    M = SEREP.reduce_dofs(M, num_nodes, [5])
    k = SEREP.reduce_dofs(k, num_nodes, [5])

    # Transform the mass matrix to a consistent mass matrix
    M = SEREP.transform_mass_matrix(M, beta=0)

    # Determine master and slave degrees of freedom
    MasterDofs, SlaveDofs = SEREP.separate_dofs(num_nodes, master_nodes)

    # Perform SEREP to obtain reduced system matrices and the transformation matrix
    MR, KR, T = SEREP.SEREP_Expansion(k, M, SlaveDofs, master_nodes)

    # Read and reduce hydrodynamic data for system matrices
    added_mass = SEREP.reduce_dofs(dataset['added_mass'][0].values, 10, [5])
    radiation_damping = SEREP.reduce_dofs(dataset['radiation_damping'][0].values, 10, [5])
    hydrostatic_stiffness = SEREP.reduce_dofs(dataset['hydrostatic_stiffness'].values, 10, [5])
    F_w_hydro = dataset['Froude_Krylov_force'][0].values + dataset['diffraction_force'][0].values
    F_w_hydro_redu = SEREP.reduce_force_matrix_dofs(F_w_hydro, 10, 5).reshape(1, 50)

    # Combine reduced hydrodynamic matrices to form system matrices
    mass = T.T @ added_mass @ T + MR
    damping = T.T @ radiation_damping @ T
    stiffness = T.T @ hydrostatic_stiffness @ T + KR
    F_w = T.T @ F_w_hydro_redu.T

    # Solve the system in the frequency domain
    master_displacement = DM_A.solve_frequency_domain(mass, damping, stiffness, F_w.T, omega)

    # Reorder the global displacement according to the master and slave DOFs
    global_displacement_disorder = master_displacement
    global_displacement = SEREP.reorder_displacement_matrix(global_displacement_disorder, MasterDofs, SlaveDofs)

    return global_displacement